# Ajuster la voix pidgin sur Common Voice camerounais (Colab)

La voix de base (MMS-TTS pcm) parle **pidgin nigérian**. On l'ajuste sur 12 voix camerounaises de Common Voice pour rapprocher la prononciation. La voix finale est un **mélange** des voix (aucun contributeur n'est reproduit).

**Avant de lancer :**
1. **Exécution → Modifier le type d'exécution** : **GPU T4**, et pour la **version de l'environnement d'exécution**, prends la version précédente (pas « Latest ») : il faut **Python 3.12**. Sous Python 3.13, `tokenizers` ne s'installe pas.
2. Le fichier **`KAMA_tts_wes_drive.zip`** (210 Mo, ne pas le décompresser) doit être à un de ces endroits :
   - **le plus sûr** : glissé dans le panneau 📁 de gauche (il arrive dans `/content`), attendre la fin de l'envoi ;
   - ou dans ton Drive (racine de Mon Drive ou sous `KAMA/`), le Drive étant monté par l'icône Drive du panneau 📁 ;
   - sinon, la cellule 2 propose un envoi manuel.
   Il contient des données Common Voice : session ou Drive privé seulement, aucun lien de partage.
3. Lance d'abord avec `SMOKE = True` (quelques minutes : 6 enregistrements par locuteur). Si tout passe jusqu'à la fin, note le **temps par pas**, mets `SMOKE = False`, relance la cellule 1, puis à partir de la section 5.

**Durée du vrai entraînement** ≈ 284 pas par passe × `EPOCHS` × temps par pas. Rien n'est enregistré avant la fin : si l'estimation dépasse 3 h, baisse `EPOCHS` (par exemple 15). Garde l'onglet ouvert et actif.

Licence : le modèle de base est CC-BY-NC 4.0 (non commercial) ; le modèle ajusté en hérite.

## 1. Réglages

In [ ]:
import sys
assert sys.version_info[:2] <= (3, 12), f'Python {sys.version.split()[0]} : change la version de l environnement d execution (Python 3.12 attendu)'
print('Python', sys.version.split()[0], 'OK')

MON_DRIVE = '/content/drive/MyDrive'
SMOKE = True      # True : essai rapide (6 enregistrements par locuteur). False : vrai entraînement.
EPOCHS = 30       # passes du vrai entraînement (environ 284 pas par passe)
LOCAL = '/content/KAMA/tts_wes'
SMOKE_DATA = '/content/smoke_data'      # petit jeu de données de l'essai
REPO = '/content/finetune-hf-vits'
BASE = '/content/mms-tts-pcm-train'     # modèle de base converti pour l'entraînement
SORTIE = '/content/tts_wes_sortie'      # sortie de l'entraînement (disque Colab)
SORTIE_DRIVE = MON_DRIVE + '/KAMA/tts_wes_sorties'   # seuls le modèle final et les échantillons sont copiés ici
print('mode', 'ESSAI (SMOKE)' if SMOKE else f'ENTRAINEMENT COMPLET, {EPOCHS} passes')

## 2. Données
Cherche le zip dans la session (`/content`), puis sur le Drive, puis propose un envoi manuel. Le zip est décompressé sur le disque de Colab.

In [ ]:
import os, glob, shutil, zipfile
os.chdir('/content')  # le montage echoue si le dossier courant est dans /content/drive

# 1) Zip deja envoye dans la session (glisse dans le panneau de gauche)
zips = glob.glob('/content/KAMA_tts_wes_drive.zip')

# 2) Sinon, le Drive (deja monte par le panneau de gauche, ou montage ici)
if not zips:
    try:
        if not os.path.isdir(MON_DRIVE):
            from google.colab import drive
            drive.mount('/content/drive', timeout_ms=600000)
        zips = glob.glob(MON_DRIVE + '/KAMA_tts_wes_drive.zip') + glob.glob(MON_DRIVE + '/KAMA/**/KAMA_tts_wes_drive.zip', recursive=True)
    except Exception as e:  # TIMEOUT : autorisation Google non aboutie
        print('Drive non monté :', type(e).__name__, '-> envoi manuel du zip')

# 3) En dernier recours, envoi depuis ton poste (garde l'onglet au premier plan)
if not zips:
    from google.colab import files
    up = files.upload()
    assert 'KAMA_tts_wes_drive.zip' in up, 'KAMA_tts_wes_drive.zip attendu'
    zips = ['/content/KAMA_tts_wes_drive.zip']

print('zip utilisé :', zips[0])
shutil.rmtree(LOCAL, ignore_errors=True)
with zipfile.ZipFile(zips[0]) as z:
    assert z.testzip() is None, 'zip abîmé : renvoie-le depuis ton ordinateur'
    z.extractall('/content')
for f in ('data/train/metadata.csv', 'data/test/metadata.csv', 'data/ecoute_phrases.json'):
    assert os.path.exists(f'{LOCAL}/{f}'), f'manquant dans le zip : {f}'

# La colonne du locuteur s'appelle « locuteur », pas « speaker_id » : le script crée lui-même une colonne speaker_id
# (numéros 0..N-1) ; si elle existe déjà dans les données, elle reste masquée et l'entraînement plante (KeyError 'speaker_id').
import pandas as pd
def lire(split):
    p = f'{LOCAL}/data/{split}/metadata.csv'
    df = pd.read_csv(p, dtype=str, keep_default_na=False).rename(columns={'speaker_id': 'locuteur'})
    df.to_csv(p, index=False)
    return df
train, test = lire('train'), lire('test')
print('Données prêtes :', len(train), "enregistrements d'entraînement,", len(test), 'de test,', train.locuteur.nunique(), 'locuteurs')

## 2 bis. Jeu de données de l'essai (SMOKE seulement)
Les fichiers audio sont lus dans l'ordre de leur nom, qui commence par le code du locuteur : prendre « les N premiers » ne donnerait qu'un seul locuteur. L'essai prend donc 6 enregistrements **par locuteur**.

In [ ]:
shutil.rmtree(SMOKE_DATA, ignore_errors=True)
if SMOKE:
    for split, df in (('train', train), ('test', test)):
        src = f'{LOCAL}/data/{split}'
        part = df.sample(frac=1, random_state=0).groupby('locuteur').head(6) if split == 'train' else df.head(8)
        os.makedirs(f'{SMOKE_DATA}/{split}/audio', exist_ok=True)
        for f in part.file_name:
            shutil.copy(f'{src}/{f}', f'{SMOKE_DATA}/{split}/{f}')
        part.to_csv(f'{SMOKE_DATA}/{split}/metadata.csv', index=False)
        print(split, len(part), 'enregistrements,', part.locuteur.nunique(), 'locuteurs')
else:
    print('Vrai entraînement : toutes les données, pas de jeu réduit.')

## 3. Installation
Outil d'ajustement de Hugging Face pour MMS-TTS (`ylacombe/finetune-hf-vits`). Versions fixées (transformers 4.x : la v5 a supprimé des réglages utilisés ici). Si `pip` signale un conflit, redémarre l'exécution (Exécution → Redémarrer) puis relance les cellules 1, 2, 2 bis et celle-ci.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets[audio]==2.21.0" "accelerate==0.34.2" soundfile librosa cython tensorboard
!rm -rf {REPO} && git clone -q https://github.com/ylacombe/finetune-hf-vits {REPO}
# matplotlib >= 3.10 n'a plus tostring_rgb (graphiques de validation) : on lit le tampon RGBA à la place
!sed -i 's/np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)/np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()/g' {REPO}/utils/plot.py
%cd {REPO}/monotonic_align
!mkdir -p monotonic_align && python setup.py build_ext --inplace
%cd {REPO}
!nvidia-smi -L

## 4. Modèle de base pour l'entraînement
Convertit le discriminateur de MMS-TTS pcm (nécessaire à l'entraînement, téléchargé depuis `facebook/mms-tts`, environ 560 Mo) et l'assemble avec la voix de base.

In [ ]:
%cd {REPO}
!python convert_original_discriminator_checkpoint.py --language_code pcm --pytorch_dump_folder_path {BASE}
print(os.listdir(BASE))

## 5. Entraînement
Dans la sortie, cherche la ligne **« Resize speaker embeddings from 1 to 12 »** : si tu vois « Only one speaker detected », arrête et vérifie la cellule 2 bis.
En mode `SMOKE`, note le **temps par pas** (`s/it`) dans la barre `Steps`.
Après le vrai entraînement, vérifie que les pertes (`loss`) ont baissé et ne sont pas devenues `nan`.

In [ ]:
import json
DONNEES = SMOKE_DATA if SMOKE else f'{LOCAL}/data'
cfg = {
    'project_name': 'kama_tts_wes', 'push_to_hub': False, 'overwrite_output_dir': True, 'output_dir': SORTIE,
    'dataset_name': DONNEES, 'audio_column_name': 'audio', 'text_column_name': 'text',
    'train_split_name': 'train', 'eval_split_name': 'test',
    'speaker_id_column_name': 'locuteur', 'override_speaker_embeddings': True,
    'max_duration_in_seconds': 12, 'min_duration_in_seconds': 1.5, 'max_tokens_length': 450,
    'model_name_or_path': BASE, 'preprocessing_num_workers': 2,
    'do_train': True, 'num_train_epochs': 1 if SMOKE else EPOCHS, 'gradient_accumulation_steps': 1, 'gradient_checkpointing': False,
    'per_device_train_batch_size': 16, 'learning_rate': 2e-5, 'adam_beta1': 0.8, 'adam_beta2': 0.99, 'warmup_ratio': 0.01, 'group_by_length': False,
    'do_eval': True, 'eval_steps': 2 if SMOKE else 284, 'per_device_eval_batch_size': 8, 'max_eval_samples': 8 if SMOKE else 40,
    'do_step_schedule_per_epoch': True,
    'weight_disc': 3, 'weight_fmaps': 1, 'weight_gen': 1, 'weight_kl': 1.5, 'weight_duration': 1, 'weight_mel': 35,
    'fp16': True, 'seed': 456, 'report_to': 'tensorboard',
}
json.dump(cfg, open('/content/config_tts.json', 'w'), indent=1)
os.environ['WANDB_DISABLED'] = 'true'

n_loc = pd.read_csv(f'{DONNEES}/train/metadata.csv', dtype=str).locuteur.nunique()
assert n_loc > 1, f'{n_loc} locuteur dans {DONNEES} : relance les cellules 2 et 2 bis'
print('mode', 'ESSAI (SMOKE)' if SMOKE else f'ENTRAINEMENT COMPLET, {EPOCHS} passes')
print('données :', DONNEES, '|', n_loc, 'locuteurs')

In [ ]:
%cd {REPO}
!accelerate launch run_vits_finetuning.py /content/config_tts.json

## 6. Voix « mélange »
Le modèle a une voix par contributeur. On ajoute une voix supplémentaire égale à la **moyenne** de toutes : c'est celle que le serveur utilisera, pour qu'aucun contributeur ne soit reproduit. Le numéro à mettre dans `routes.json` est affiché à la fin.

In [ ]:
import torch
from transformers import VitsModel, AutoTokenizer
m = VitsModel.from_pretrained(SORTIE)
tok = AutoTokenizer.from_pretrained(SORTIE)
assert m.config.num_speakers > 1, 'le modèle entraîné n a qu une voix : vérifie la section 5'
w = m.embed_speaker.weight.data.clone()
n = w.shape[0]
m.embed_speaker = torch.nn.Embedding(n + 1, w.shape[1])
m.embed_speaker.weight.data = torch.cat([w, w.mean(dim=0, keepdim=True)])
m.config.num_speakers = n + 1
MIX_ID = n
FINAL = '/content/mms-tts-wes-v1'
shutil.rmtree(FINAL, ignore_errors=True)
m.save_pretrained(FINAL)
tok.save_pretrained(FINAL)
print(f'{n} voix + la voix mélange. speaker_id du mélange = {MIX_ID}  <- à mettre dans routes.json (options.speaker_id)')
print(os.listdir(FINAL))

## 7. Échantillons à écouter
Pour chaque phrase : `_base` (voix nigériane d'origine) et `_ajuste` (voix ajustée, mélange). Les phrases viennent des textes validés de la page de test et de phrases de test jamais apprises.

In [ ]:
import re, scipy.io.wavfile as wav
phrases = json.load(open(f'{LOCAL}/data/ecoute_phrases.json', encoding='utf-8'))
vocab = set(json.load(open(f'{FINAL}/vocab.json')))
def propre(t):
    return re.sub(r'\s+', ' ', ''.join(c if c in vocab else ' ' for c in t.lower())).strip()
base = VitsModel.from_pretrained('facebook/mms-tts-pcm').eval()
tok0 = AutoTokenizer.from_pretrained('facebook/mms-tts-pcm')
m.eval()
shutil.rmtree('/content/ecoute', ignore_errors=True)
os.makedirs('/content/ecoute')
for i, t in enumerate(phrases):
    texte = propre(t)
    if not texte:
        continue
    with torch.inference_mode():
        torch.manual_seed(0)
        a = base(**tok0(texte, return_tensors='pt')).waveform[0].numpy()
        torch.manual_seed(0)
        b = m(**tok(texte, return_tensors='pt'), speaker_id=MIX_ID).waveform[0].numpy()
    wav.write(f'/content/ecoute/{i:02d}_base.wav', 16000, a)
    wav.write(f'/content/ecoute/{i:02d}_ajuste.wav', 16000, b)
    print(i, texte)
print(len(os.listdir('/content/ecoute')), 'fichiers dans /content/ecoute')

## 8. Récupérer le modèle
Si le Drive est monté : copie de `mms-tts-wes-v1.zip` (environ 140 Mo) et `ecoute.zip` dans `Mon Drive/KAMA/tts_wes_sorties`. Sinon : téléchargement direct dans le navigateur (garde l'onglet ouvert jusqu'à la fin).
Sur ton poste, le modèle va dans `KAMA/models/poids/tts/mms-tts-wes-v1` (hors Git).

In [ ]:
suffixe = '_essai' if SMOKE else ''
!cd /content && rm -f mms-tts-wes-v1{suffixe}.zip ecoute{suffixe}.zip && zip -qr mms-tts-wes-v1{suffixe}.zip mms-tts-wes-v1 && zip -qr ecoute{suffixe}.zip ecoute
if os.path.isdir(MON_DRIVE):
    os.makedirs(SORTIE_DRIVE, exist_ok=True)
    shutil.copy(f'/content/mms-tts-wes-v1{suffixe}.zip', SORTIE_DRIVE)
    shutil.copy(f'/content/ecoute{suffixe}.zip', SORTIE_DRIVE)
    print(os.listdir(SORTIE_DRIVE))
else:
    from google.colab import files
    files.download(f'/content/mms-tts-wes-v1{suffixe}.zip')
    files.download(f'/content/ecoute{suffixe}.zip')
    print('zips téléchargés : range-les dans Mon Drive/KAMA/tts_wes_sorties')